# Week 3 Assignment 1
## Exploratory Data Analysis (EDA) and Machine Learning on Agricultural Yield Dataset

**Name:** Kisa  
**Enrollment No:** 03404092025  
**Course:** ML and Generative AI with Python

## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Display settings
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (8, 5)
sns.set_theme(style='whitegrid')

---
## Part A: Understanding the Dataset

### Q1. Dataset Overview

In [ ]:
# Load the dataset
df = pd.read_csv('agriculture_yield_dataset.csv')

print("Number of Rows and Columns:", df.shape)
print(f"  → {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
print("Column Names:")
for col in df.columns:
    print(" -", col)

In [ ]:
print("First 10 Records:")
df.head(10)

### Q2. Data Types and Missing Values

In [ ]:
print("Data Types of Each Column:")
print(df.dtypes)

In [ ]:
print("Missing Values in Each Column:")
missing = df.isnull().sum()
print(missing)
if missing.sum() == 0:
    print("\n✅ No missing values found in the dataset.")
else:
    print("\nColumns with missing values:", missing[missing > 0].index.tolist())

### Q3. Descriptive Statistics

In [ ]:
print("Summary Statistics for Numerical Features:")
df.describe()

In [ ]:
desc = df.describe()
highest_mean_col = desc.loc['mean'].idxmax()
highest_mean_val = desc.loc['mean'].max()
highest_std_col  = desc.loc['std'].idxmax()
highest_std_val  = desc.loc['std'].max()

print(f"Feature with Highest Mean : '{highest_mean_col}' → {highest_mean_val:.2f}")
print(f"Feature with Highest Std  : '{highest_std_col}' → {highest_std_val:.2f}")

---
## Part B: Exploratory Data Analysis (EDA)

### Q4. Distribution Analysis – Histograms

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features = ['rainfall_mm', 'temperature_c', 'fertilizer_kg', 'yield_ton_per_hectare']
colors   = ['steelblue', 'tomato', 'seagreen', 'darkorange']

for ax, feat, color in zip(axes.flatten(), features, colors):
    ax.hist(df[feat], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribution of {feat}', fontsize=13)
    ax.set_xlabel(feat)
    ax.set_ylabel('Frequency')

plt.suptitle('Histograms of Key Numerical Features', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('histograms.png', bbox_inches='tight', dpi=150)
plt.show()

**Observations:**

**rainfall_mm:**
1. The distribution is fairly uniform/slightly right-skewed, ranging from ~300 to 1200 mm.
2. No extreme concentration at any single peak, indicating diverse climatic conditions in the dataset.
3. Most values cluster between 500–1000 mm.

**temperature_c:**
1. Approximately uniform distribution across 18°C to 38°C.
2. No strong peak, suggesting data was collected across multiple seasons or regions.
3. Central temperatures (25–32°C) are slightly more frequent.

**fertilizer_kg:**
1. Roughly uniform spread from ~50 to 300 kg, with no single dominant value.
2. Usage is spread across a wide range, indicating varied farming practices.
3. Slight density around 100–250 kg range.

**yield_ton_per_hectare:**
1. Approximately bell-shaped (near-normal), centred around 5 tons/hectare.
2. Most yields fall between 3.5 and 6.5 tons/hectare.
3. Very few extreme low or high yield values, suggesting a stable distribution.

### Q5. Crop Type Analysis

In [ ]:
crop_counts = df['crop_type'].value_counts()
print("Records per Crop Type:")
print(crop_counts)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(crop_counts.index, crop_counts.values,
              color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'],
              edgecolor='white', width=0.6)
ax.bar_label(bars, padding=3, fontsize=11)
ax.set_title('Count of Records per Crop Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Crop Type')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('crop_type_countplot.png', dpi=150)
plt.show()

print(f"\nMost Frequent Crop: {crop_counts.idxmax()} ({crop_counts.max()} records)")

### Q6. Soil Type Analysis

In [ ]:
soil_counts = df['soil_type'].value_counts()
print("Frequency of Each Soil Type:")
print(soil_counts)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(soil_counts.index, soil_counts.values,
              color=['#6baed6','#fd8d3c','#74c476'],
              edgecolor='white', width=0.5)
ax.bar_label(bars, padding=3, fontsize=11)
ax.set_title('Count of Records per Soil Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Soil Type')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('soil_type_countplot.png', dpi=150)
plt.show()

print(f"\nMost Common Soil Type: {soil_counts.idxmax()} ({soil_counts.max()} records)")

### Q7. Yield Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['yield_ton_per_hectare'], bins=35, color='darkorange',
        edgecolor='white', alpha=0.85)
ax.set_title('Distribution of Yield (ton/hectare)', fontsize=13, fontweight='bold')
ax.set_xlabel('yield_ton_per_hectare')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('yield_distribution.png', dpi=150)
plt.show()

mean_y  = df['yield_ton_per_hectare'].mean()
std_y   = df['yield_ton_per_hectare'].std()
skew_y  = df['yield_ton_per_hectare'].skew()
print(f"Mean  : {mean_y:.2f}")
print(f"Std   : {std_y:.2f}")
print(f"Skew  : {skew_y:.4f}  (values near 0 → approximately normal)")

**Answers:**
- **Approximately Normal?** Yes – the distribution is roughly bell-shaped and centred around ~5 tons/hectare with a skewness close to 0.
- **Noticeable Outliers?** There are a few records at the low end (<2.5) and high end (>7.5) that could be considered mild outliers, but no extreme outliers that distort the distribution significantly.

### Q8. Scatter Plot Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['rainfall_mm'], df['yield_ton_per_hectare'],
                alpha=0.35, color='steelblue', edgecolors='none', s=20)
axes[0].set_title('Rainfall vs Yield', fontsize=12, fontweight='bold')
axes[0].set_xlabel('rainfall_mm')
axes[0].set_ylabel('yield_ton_per_hectare')

axes[1].scatter(df['fertilizer_kg'], df['yield_ton_per_hectare'],
                alpha=0.35, color='seagreen', edgecolors='none', s=20)
axes[1].set_title('Fertilizer vs Yield', fontsize=12, fontweight='bold')
axes[1].set_xlabel('fertilizer_kg')
axes[1].set_ylabel('yield_ton_per_hectare')

plt.suptitle('Scatter Plots: Feature vs Yield', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150)
plt.show()

corr_rain = df['rainfall_mm'].corr(df['yield_ton_per_hectare'])
corr_fert = df['fertilizer_kg'].corr(df['yield_ton_per_hectare'])
print(f"Correlation – rainfall_mm vs yield   : {corr_rain:.4f}")
print(f"Correlation – fertilizer_kg vs yield : {corr_fert:.4f}")

**Observation:**
- `rainfall_mm` (r ≈ 0.55) shows a noticeably stronger positive relationship with yield than `fertilizer_kg` (r ≈ 0.28).
- The scatter for rainfall is more structured (upward trend visible), while fertilizer shows a wider, noisier spread.
- **Conclusion:** `rainfall_mm` appears to have the stronger relationship with crop yield.

### Q9. Correlation Analysis

In [ ]:
num_df = df.select_dtypes(include='number')
corr_matrix = num_df.corr()

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, annot_kws={'size': 11})
ax.set_title('Correlation Heatmap of Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

print("\nCorrelation with yield_ton_per_hectare (sorted):")
top_corr = corr_matrix['yield_ton_per_hectare'].drop('yield_ton_per_hectare').abs().sort_values(ascending=False)
print(top_corr)
print("\nTop 3 Features Most Correlated with Yield:")
for i, (feat, val) in enumerate(top_corr.head(3).items(), 1):
    print(f"  {i}. {feat} → |r| = {val:.4f}")

### Q10. Group-Based Analysis

In [ ]:
avg_crop = df.groupby('crop_type')['yield_ton_per_hectare'].mean().sort_values(ascending=False)
avg_soil = df.groupby('soil_type')['yield_ton_per_hectare'].mean().sort_values(ascending=False)

print("Average Yield by Crop Type:")
print(avg_crop.round(2))
print()
print("Average Yield by Soil Type:")
print(avg_soil.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(avg_crop.index, avg_crop.values, color='#4C72B0', edgecolor='white')
axes[0].set_title('Avg Yield by Crop Type', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Crop Type')
axes[0].set_ylabel('Avg Yield (ton/ha)')
for i, v in enumerate(avg_crop.values):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)

axes[1].bar(avg_soil.index, avg_soil.values, color='#55A868', edgecolor='white')
axes[1].set_title('Avg Yield by Soil Type', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Soil Type')
axes[1].set_ylabel('Avg Yield (ton/ha)')
for i, v in enumerate(avg_soil.values):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('group_yield.png', dpi=150)
plt.show()

print(f"\nHighest Average Yield → Crop Type : {avg_crop.idxmax()} ({avg_crop.max():.2f} ton/ha)")
print(f"Highest Average Yield → Soil Type : {avg_soil.idxmax()} ({avg_soil.max():.2f} ton/ha)")

---
## Part C: Data Preparation

### Q11. Feature Encoding

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
print("Categorical Columns Identified:")
for c in cat_cols:
    print(f"  - {c}  (unique values: {df[c].unique().tolist()})")

In [ ]:
# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['crop_type', 'soil_type'], drop_first=False)

print("Shape after encoding:", df_encoded.shape)
print("\nNew Columns Added:")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print(new_cols)

print("\nFirst 5 Rows of Transformed Dataset:")
df_encoded.head()

### Q12. Feature Selection

In [ ]:
# Target variable
TARGET = 'yield_ton_per_hectare'

X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f"Target Variable (y): '{TARGET}'")
print(f"\nInput Features (X) — {X.shape[1]} columns:")
print(list(X.columns))
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

---
## Part D: Machine Learning

### Q13. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train-Test Split (80% / 20%):")
print(f"  X_train shape : {X_train.shape}")
print(f"  X_test  shape : {X_test.shape}")
print(f"  y_train shape : {y_train.shape}")
print(f"  y_test  shape : {y_test.shape}")

### Q14. Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Model Intercept: {model.intercept_:.4f}")
print()

coef_df = pd.Series(model.coef_, index=X.columns).sort_values(ascending=False)
print("Model Coefficients (sorted descending):")
print(coef_df.round(6).to_string())

In [ ]:
# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['seagreen' if v > 0 else 'tomato' for v in coef_df.values]
ax.barh(coef_df.index, coef_df.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression Coefficients', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('coefficients.png', dpi=150)
plt.show()

print(f"\nFeature with Highest Positive Coefficient: '{coef_df.idxmax()}' ({coef_df.max():.4f})")
print()
print("Interpretation: crop_type_Rice has the highest positive coefficient,")
print("meaning Rice crops are most strongly associated with higher yield predictions,")
print("holding all other features constant.")